# SAM 3 & Open-Vocabulary Segmentation Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Prompt construction

Build a helper that turns a user sentence into a list of SAM 3 concept prompts. This is the boundary where "what the user typed" meets "what the model consumes".

In [ ]:
```python

def split_concepts(sentence):

    """

    Heuristic splitter for multi-concept prompts.

    Returns list of short noun phrases.

    """

    for sep in [",", ";", "and", "or", "&"]:

        if sep in sentence:

            parts = [p.strip() for p in sentence.replace("and ", ",").split(",")]

            return [p for p in parts if p]

    return [sentence.strip()]

print(split_concepts("cats, dogs and balloons"))

In [ ]:
```

SAM 3 accepts one concept per forward pass; for multi-concept queries, loop or batch them.

### Step 2: Post-processing helpers

Turn SAM 3's raw outputs into a clean list of detections that match our Phase 4 Lesson 16 pipeline contract.

In [ ]:
```python

from dataclasses import dataclass

from typing import List

@dataclass

class ConceptDetection:

    concept: str

    instance_id: int

    box: tuple          # (x1, y1, x2, y2)

    score: float

    mask_rle: str       # run-length encoded

def rle_encode(binary_mask):

    flat = binary_mask.flatten().astype("uint8")

    runs = []

    prev, count = flat[0], 0

    for v in flat:

        if v == prev:

            count += 1

        else:

            runs.append((int(prev), count))

            prev, count = v, 1

    runs.append((int(prev), count))

    return ";".join(f"{v}x{c}" for v, c in runs)

In [ ]:
```

RLE keeps response payloads small even for many high-resolution masks. The same format works across SAM 2, SAM 3, Grounded SAM 2.

### Step 3: A unified open-vocab segmentation interface

Wrap whatever backend you have (SAM 3, Grounded SAM 2, YOLO-World + SAM 2) behind a single method. Your downstream code does not change when the backend does.

In [ ]:
```python

from abc import ABC, abstractmethod

import numpy as np

class OpenVocabSeg(ABC):

    @abstractmethod

    def detect(self, image: np.ndarray, concept: str) -> List[ConceptDetection]:

        ...

class StubOpenVocabSeg(OpenVocabSeg):

    """

    Deterministic stub used for pipeline testing when real models are not loaded.

    """

    def detect(self, image, concept):

        h, w = image.shape[:2]

        return [

            ConceptDetection(

                concept=concept,

                instance_id=0,

                box=(w * 0.2, h * 0.3, w * 0.5, h * 0.8),

                score=0.89,

                mask_rle="0x100;1x50;0x200",

            ),

            ConceptDetection(

                concept=concept,

                instance_id=1,

                box=(w * 0.55, h * 0.25, w * 0.85, h * 0.75),

                score=0.74,

                mask_rle="0x80;1x40;0x220",

            ),

        ]

In [ ]:
```

The real `SAM3OpenVocabSeg` subclass would wrap `transformers.Sam3Model` and `Sam3Processor`.

### Step 4: Hugging Face SAM 3 usage (reference)

For the actual model, the `transformers` integration:

In [ ]:
```python

from transformers import Sam3Processor, Sam3Model

import torch

processor = Sam3Processor.from_pretrained("facebook/sam3")

model = Sam3Model.from_pretrained("facebook/sam3").eval()

inputs = processor(images=pil_image, return_tensors="pt")

inputs = processor.set_text_prompt(inputs, "yellow school bus")

with torch.no_grad():

    outputs = model(**inputs)

masks = processor.post_process_masks(

    outputs.masks, inputs.original_sizes, inputs.reshaped_input_sizes

)

boxes = outputs.boxes

scores = outputs.scores

In [ ]:
```

One prompt, all matches returned in a single call.

### Step 5: Measure what Grounded SAM 2 gave you for free

An honest benchmark: what happens when you replace Grounded SAM 2 with SAM 3 in a real pipeline?

- Latency: SAM 3 saves one forward pass (no separate detector) but the model itself is heavier; usually net-neutral or a slight speedup.

- Accuracy: SAM 3 substantially better on rare or compositional concepts ("striped red umbrella"). Similar on common single-word concepts.

- Flexibility: Grounded SAM 2 lets you swap detectors (DINO-X, Florence-2, Grounding DINO 1.5); SAM 3 is monolithic.

Conclusion: SAM 3 is the default for 2026 open-vocab seg. Grounded SAM 2 is still the right answer when you need detector flexibility or different license terms.

## Exercises

In [ ]:
1. **(Easy)** Run SAM 3 on 10 images with concept prompts you choose. Compare against SAM 2 + Grounding DINO 1.5 on the same images. Report which concepts each model missed.
2. **(Medium)** Build a "click-to-include / click-to-exclude" UI on top of SAM 3: a text prompt returns candidate instances; user clicks keep which ones count as positive. Output the final concept set as JSON.
3. **(Hard)** Fine-tune SAM 3 on a custom concept set (e.g. 5 types of electronic components) with 20 labelled images each. Compare to zero-shot SAM 3 on the same test set; measure mask IoU improvement.